# Lab | Web Scraping

Welcome to the "Books to Scrape" Web Scraping Adventure Lab!

**Objective**

In this lab, we will embark on a mission to unearth valuable insights from the data available on Books to Scrape, an online platform showcasing a wide variety of books. As data analyst, you have been tasked with scraping a specific subset of book data from Books to Scrape to assist publishing companies in understanding the landscape of highly-rated books across different genres. Your insights will help shape future book marketing strategies and publishing decisions.

**Background**

In a world where data has become the new currency, businesses are leveraging big data to make informed decisions that drive success and profitability. The publishing industry, much like others, utilizes data analytics to understand market trends, reader preferences, and the performance of books based on factors such as genre, author, and ratings. Books to Scrape serves as a rich source of such data, offering detailed information about a diverse range of books, making it an ideal platform for extracting insights to aid in informed decision-making within the literary world.

**Task**

Your task is to create a Python script using BeautifulSoup and pandas to scrape Books to Scrape book data, focusing on book ratings and genres. The script should be able to filter books with ratings above a certain threshold and in specific genres. Additionally, the script should structure the scraped data in a tabular format using pandas for further analysis.

**Expected Outcome**

A function named `scrape_books` that takes two parameters: `min_rating` and `max_price`. The function should scrape book data from the "Books to Scrape" website and return a `pandas` DataFrame with the following columns:

**Expected Outcome**

- A function named `scrape_books` that takes two parameters: `min_rating` and `max_price`.
- The function should return a DataFrame with the following columns:
  - **UPC**: The Universal Product Code (UPC) of the book.
  - **Title**: The title of the book.
  - **Price (£)**: The price of the book in pounds.
  - **Rating**: The rating of the book (1-5 stars).
  - **Genre**: The genre of the book.
  - **Availability**: Whether the book is in stock or not.
  - **Description**: A brief description or product description of the book (if available).
  
You will execute this script to scrape data for books with a minimum rating of `4.0 and above` and a maximum price of `£20`. 

Remember to experiment with different ratings and prices to ensure your code is versatile and can handle various searches effectively!

**Resources**

- [Beautiful Soup Documentation](https://www.crummy.com/software/BeautifulSoup/bs4/doc/)
- [Pandas Documentation](https://pandas.pydata.org/pandas-docs/stable/index.html)
- [Books to Scrape](https://books.toscrape.com/)


**Hint**

Your first mission is to familiarize yourself with the **Books to Scrape** website. Navigate to [Books to Scrape](http://books.toscrape.com/) and explore the available books to understand their layout and structure. 

Next, think about how you can set parameters for your data extraction:

- **Minimum Rating**: Focus on books with a rating of 4.0 and above.
- **Maximum Price**: Filter for books priced up to £20.

After reviewing the site, you can construct a plan for scraping relevant data. Pay attention to the details displayed for each book, including the title, price, rating, and availability. This will help you identify the correct HTML elements to target with your scraping script.

Make sure to build your scraping URL and logic based on the patterns you observe in the HTML structure of the book listings!


---

**Best of luck! Immerse yourself in the world of books, and may the data be with you!**

**Important Note**:

In the fast-changing online world, websites often update and change their structures. When you try this lab, the **Books to Scrape** website might differ from what you expect.

If you encounter issues due to these changes, like new rules or obstacles preventing data extraction, don’t worry! Get creative.

You can choose another website that interests you and is suitable for scraping data. Options like Wikipedia, The New York Times, or even library databases are great alternatives. The main goal remains the same: extract useful data and enhance your web scraping skills while exploring a source of information you enjoy. This is your opportunity to practice and adapt to different web environments!

In [63]:
import pandas as pd
from bs4 import BeautifulSoup
import requests

In [ ]:
url = "https://books.toscrape.com/"
 # requests.get(url): Python lanza una llamada al servidor web (como cuando tú escribes la dirección en Chrome). El servidor responde enviando todo el código HTML de la página.
response = requests.get(url)
# response.content: Es el código HTML "en bruto" (texto puro y difícil de leer). 
# BeautifulSoup(..., "html.parser"): Toma ese texto bruto y lo convierte en un "árbol" organizado. Ahora Python puede decir: "búscame el título" en lugar de leer miles de líneas de texto.
soup = BeautifulSoup(response.content, "html.parser") 
# find_all: Busca todas las etiquetas HTML que cumplan el requisito.
# 'article', class_='product_pod': En esta web, cada libro está encerrado en una "cajita". Busca todos los encabezados <article> de la clase 'product_pod'. cards ahora es una lista que contiene las 20 cajitas de libros de esa página.
cards = soup.find_all('article', class_='product_pod')

In [97]:
def scrape_books(rating_text, max_price):
    all_books = []
    
    print("Iniciando extracción de datos...")

    # Recorremos las primeras 3 páginas (puedes aumentar el rango si quieres más datos)
    for i in range(10, 11):
        url = f'https://books.toscrape.com/catalogue/page-{i}.html'
        response = requests.get(url)
        soup = BeautifulSoup(response.content, "html.parser")
        cards = soup.find_all('article', class_='product_pod')
        print(f'Extrayendo información página {i}...')

        for c in cards:
            # 1. Extraer datos básicos de la tarjeta
            title = c.find('h3').find('a')['title'] # Busca el encabezado <h3>, luego el enlace <a> y extrae el atributo 'title'
            price = float(c.find('p', class_='price_color').text.replace('£', '')) # .text: Saca solo el texto (ej: "£15.50").
            rating = c.find('p', class_='star-rating').get('class')[1]

            # 2. Aplicar filtros iniciales (Precio y Rating)
            if price <= max_price and rating_text in ('Four', 'Five'):
                
                # 3. Obtener URL del detalle para extraer UPC, Género y Descripción
                relative_url = c.find('h3').find('a')['href'].replace('catalogue/', '')
                book_url = 'https://books.toscrape.com/catalogue/' + relative_url
                
                try:
                    book_res = requests.get(book_url)
                    book_soup = BeautifulSoup(book_res.content, "html.parser")
                    
                    # Extraer UPC y Disponibilidad de la tabla técnica
                    table = book_soup.find('table', class_='table table-striped')
                    rows = table.find_all('tr')
                    upc = rows[0].find('td').text
                    availability = rows[5].find('td').text
                    
                    # Extraer Género (está en los breadcrumbs)
                    genre_tag = book_soup.find('ul', class_='breadcrumb')
                    genre = genre_tag.find_all('li')[2].text.strip() if (genre_tag and len(genre_tag.find_all('li')) >= 3) else "Sin genero"
                    
                    # Extraer Descripción (está en el párrafo después del ID product_description)
                    desc_tag = book_soup.find('div', class_='sub-header')
                    description = desc_tag.find_next('p').text if desc_tag else "Sin descripción"

                    # 4. Guardar todo en un diccionario
                    all_books.append({
                        'UPC': upc,
                        'Title': title,
                        'Genre': genre,
                        'Price (£)': price,
                        'Rating': rating,
                        'Availability': availability,
                        'Description': description
                    })
                except Exception as e:
                    print(f"Error al entrar al libro '{title}': {e}")

    # Crear el DataFrame
    df = pd.DataFrame(all_books)
    return df 

In [98]:
df = scrape_books('Four', 20)
df.head()


Iniciando extracción de datos...
Extrayendo información página 10...


,UPC,Title,Genre,Price (£),Rating,Availability,Description
0,af96613fdd793a3c,Miss Peregrine’s Home for Peculiar Children (M...,Default,10.76,One,In stock (15 available),A mysterious island. An abandoned orphanage. A...
1,d69dd8dac66f9f84,Louisa: The Extraordinary Life of Mrs. Adams,Biography,16.85,Two,In stock (15 available),An intimate portrait of Louisa Catherine Adams...
2,4aa03792b50f0b22,Little Red,Childrens,13.47,Three,In stock (15 available),"On her way to Grandmas house, Little Red Ridin..."
3,c1afd71fc86bf682,Large Print Heart of the Pride,Default,19.15,Two,In stock (15 available),Paranormal Romance Erotic Werelion Thrill Ride...


In [ ]:
df['Description']

0     A mysterious island. An abandoned orphanage. A...
1     An intimate portrait of Louisa Catherine Adams...
2     On her way to Grandmas house, Little Red Ridin...
3     Paranormal Romance Erotic Werelion Thrill Ride...
4     They call me a slut. Maybe I am.Sometimes I do...
5     Researcher and thought leader Dr. Brené Brown ...
6     While volunteering at a youth camp in the Ozar...
7     This masterpiece of modern comics storytelling...
8     In 1959 when her mother dies, twelve-year-old ...
9     All major religions have saints and sages that...
10    A widely admired writer on religion celebrates...
11    The summer before Ivy’s senior year is going t...
Name: Description, dtype: str

"A mysterious island. An abandoned orphanage. A strange collection of curious photographs.A horrific family tragedy sets sixteen-year-old Jacob journeying to a remote island off the coast of Wales, where he discovers the crumbling ruins of Miss Peregrine’s Home for Peculiar Children. As Jacob explores its abandoned bedrooms and hallways, it becomes clear that the children w A mysterious island. An abandoned orphanage. A strange collection of curious photographs.A horrific family tragedy sets sixteen-year-old Jacob journeying to a remote island off the coast of Wales, where he discovers the crumbling ruins of Miss Peregrine’s Home for Peculiar Children. As Jacob explores its abandoned bedrooms and hallways, it becomes clear that the children were more than just peculiar. They may have been dangerous. They may have been quarantined on a deserted island for good reason. And somehow—impossible though it seems—they may still be alive.A spine-tingling fantasy illustrated with haunting vintag